In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Colab\ Notebooks/fraud-detection

import os
os.makedirs("plots", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

!pip install xgboost lightgbm -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix
)
from sklearn.utils import compute_class_weight

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
df = pd.read_csv("creditcard.csv")

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

weights = dict(zip(np.unique(y_train), class_weights))

ratio = (y_train == 0).sum() / (y_train == 1).sum()


In [ ]:
models = {

    "XGBoost_tuned": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=ratio,
        tree_method="hist",
        eval_metric="logloss",
        random_state=42
    ),

    "LightGBM_tuned": LGBMClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        class_weight="balanced",
        random_state=42
    )
}


In [ ]:
def plot_roc_curve(model_name, y_true, y_probs):
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    roc_auc = roc_auc_score(y_true, y_probs)

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - {model_name}")
    plt.legend()
    plt.savefig(f"plots/roc_{model_name}.png")
    plt.show()


def plot_pr_curve(model_name, y_true, y_probs):
    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    pr_auc = average_precision_score(y_true, y_probs)

    plt.figure()
    plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve - {model_name}")
    plt.legend()
    plt.savefig(f"plots/pr_{model_name}.png")
    plt.show()


def plot_confusion_heatmap(model_name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.savefig(f"plots/cm_{model_name}.png")
    plt.show()


def plot_prob_dist(model_name, y_probs, y_true):
    plt.figure()
    plt.hist(y_probs[y_true == 0], bins=50, alpha=0.6, label="Class 0")
    plt.hist(y_probs[y_true == 1], bins=50, alpha=0.6, label="Class 1")
    plt.xlabel("Predicted Probability")
    plt.ylabel("Count")
    plt.title(f"Probability Distribution - {model_name}")
    plt.legend()
    plt.savefig(f"plots/prob_{model_name}.png")
    plt.show()

In [ ]:
results = []

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)

    y_probs = model.predict_proba(X_test)[:, 1]
    y_preds = model.predict(X_test)

    roc = roc_auc_score(y_test, y_probs)
    pr = average_precision_score(y_test, y_probs)

    results.append([name, roc, pr])

    plot_roc_curve(name, y_test, y_probs)
    plot_pr_curve(name, y_test, y_probs)
    plot_confusion_heatmap(name, y_test, y_preds)
    plot_prob_dist(name, y_probs, y_test)

In [ ]:
experiment_df = pd.DataFrame(
    results,
    columns=["Model", "ROC_AUC", "PR_AUC"]
)

print("\nFinal Experiment Results:")
print(experiment_df)

experiment_df.to_csv("results/experiment_results.csv", index=False)

In [ ]:
xgb_model = models["XGBoost_tuned"]

importance = xgb_model.feature_importances_
features = X.columns

imp_df = pd.DataFrame({
    "Feature": features,
    "Importance": importance
}).sort_values(by="Importance", ascending=False).head(15)

plt.figure()
plt.barh(imp_df["Feature"], imp_df["Importance"])
plt.gca().invert_yaxis()
plt.title("Top 15 Feature Importance (XGBoost)")
plt.tight_layout()
plt.savefig("plots/feature_importance_xgb.png")
plt.show()